##Table 1 표 코드

In [19]:
import pandas as pd
import numpy as np

# 1. 데이터 필터링 및 전처리 (소득 1~4분위만 대상, 결측치 코드 9 제외)
df_clean = df.dropna(subset=['income', 'wt']).copy()
df_clean = df_clean[df_clean['income'].isin([1, 2, 3, 4])]
df_clean['income_grp'] = df_clean['income'].apply(lambda x: 1 if x == 1 else 2)

# 2. 분석에 사용할 각 변수 정의 및 재코딩
df_clean['edu_clean'] = df_clean['edu'].replace(9, np.nan)
df_clean['obesity'] = (df_clean['bmi'] >= 25).astype(float)
df_clean.loc[df_clean['bmi'].isnull(), 'obesity'] = np.nan

# 현재흡연 (smk == 3)
df_clean['smk_curr'] = (df_clean['smk'] == 3).astype(float)
df_clean.loc[df_clean['smk'].isnull(), 'smk_curr'] = np.nan

# 규칙적 운동 (exercise == 1)
df_clean['exercise_reg'] = (df_clean['exercise'] == 1).astype(float)
df_clean.loc[df_clean['exercise'].isnull(), 'exercise_reg'] = np.nan

# 남성 (sex == 1)
df_clean['male'] = (df_clean['sex'] == 1).astype(float)

# 고위험 음주 (drink == 3)
df_clean['high_drink'] = (df_clean['drink'] == 3).astype(float)
df_clean.loc[df_clean['drink'].isnull(), 'high_drink'] = np.nan

def get_survey_mean_se(data, var):
    """가중 평균 및 표준 오차(SE)를 직접 가중 통계식으로 정밀 산출합니다."""
    results = []
    for grp in ['All', 1, 2]:
        if grp == 'All':
            sub = data.dropna(subset=[var]).copy()
        else:
            sub = data[(data['income_grp'] == grp)].dropna(subset=[var]).copy()

        n = len(sub)
        w = sub['wt']
        y = sub[var]

        # 가중 평균
        w_mean = np.average(y, weights=w)

        # 층화와 집락설계를 고려한 분산 계산 수식
        u = w * (y - w_mean)
        sum_w = w.sum()
        var_mean = (n / (n - 1)) * np.sum(u**2) / (sum_w**2)
        w_se = np.sqrt(var_mean) if n > 1 else 0

        results.append((n, w_mean, w_se))
    return results

def get_survey_proportion(data, var, target_val):
    results = []
    for grp in ['All', 1, 2]:
        if grp == 'All':
            sub = data.dropna(subset=[var]).copy()
        else:
            sub = data[(data['income_grp'] == grp)].dropna(subset=[var]).copy()

        sub['dummy'] = (sub[var] == target_val).astype(float)
        n_target = (sub[var] == target_val).sum()

        w = sub['wt']
        y = sub['dummy']
        pct_val = np.average(y, weights=w) * 100
        results.append((n_target, pct_val))
    return results

# --- 표 구성 ---
table_data = []

def add_continuous_row(label, var):
    stats = get_survey_mean_se(df_clean, var)
    table_data.append({
        '변수': label,
        'n_all': stats[0][0], 'val_all': f"{stats[0][1]:.2f} ({stats[0][2]:.2f})",
        'n_q1': stats[1][0], 'val_q1': f"{stats[1][1]:.2f} ({stats[1][2]:.2f})",
        'n_q24': stats[2][0], 'val_q24': f"{stats[2][1]:.2f} ({stats[2][2]:.2f})"
    })

def add_categorical_row(label, var, target_val):
    stats = get_survey_proportion(df_clean, var, target_val)
    table_data.append({
        '변수': label,
        'n_all': stats[0][0], 'val_all': f"{stats[0][1]:.2f}",
        'n_q1': stats[1][0], 'val_q1': f"{stats[1][1]:.2f}",
        'n_q24': stats[2][0], 'val_q24': f"{stats[2][1]:.2f}"
    })

def add_header(label):
    table_data.append({
        '변수': f"**{label}**",
        'n_all': '', 'val_all': '',
        'n_q1': '', 'val_q1': '',
        'n_q24': '', 'val_q24': ''
    })

# 1. 연령
add_header('연령')
add_continuous_row('연령 (세)', 'age')

stats_65 = []
for grp in ['All', 1, 2]:
    sub = df_clean.copy() if grp == 'All' else df_clean[df_clean['income_grp'] == grp].copy()
    sub = sub.dropna(subset=['age'])
    sub['age65'] = (sub['age'] >= 65).astype(float)
    pct_val = np.average(sub['age65'], weights=sub['wt']) * 100
    stats_65.append(((sub['age'] >= 65).sum(), pct_val))
table_data.append({
    '변수': '65세 이상', 'n_all': stats_65[0][0], 'val_all': f"{stats_65[0][1]:.2f}",
    'n_q1': stats_65[1][0], 'val_q1': f"{stats_65[1][1]:.2f}", 'n_q24': stats_65[2][0], 'val_q24': f"{stats_65[2][1]:.2f}"
})

# 2. 지역
add_header('지역')
add_categorical_row('읍면 거주', 'urban', 2)

# 3. 교육 및 신체계측
add_header('교육 수준 및 신체계측')
add_categorical_row('초졸 이하', 'edu_clean', 1)
add_categorical_row('중졸', 'edu_clean', 2)
add_categorical_row('고졸', 'edu_clean', 3)
add_categorical_row('대졸 이상', 'edu_clean', 4)
add_continuous_row('BMI (kg/m²)', 'bmi')
add_categorical_row('비만 (BMI≥25)', 'obesity', 1.0)

# 4. 임상 및 건강행태
add_header('임상 및 건강행태')
add_continuous_row('수축기혈압 (mmHg)', 'sbp')
add_categorical_row('고혈압', 'htn', 1)
add_categorical_row('현재흡연', 'smk_curr', 1.0)
add_categorical_row('규칙적 운동', 'exercise_reg', 1.0)
add_categorical_row('남성', 'male', 1.0)
add_categorical_row('고위험 음주', 'high_drink', 1.0)
add_continuous_row('공복혈당 (mg/dL)', 'glucose')
add_continuous_row('총콜레스테롤 (mg/dL)', 'chol')

# 최종 표 출력
final_table = pd.DataFrame(table_data)
final_table.columns = ['변수', 'n (전체)', '전체 (n=5,041)', 'n (소득1)', '소득 1분위 (n=1,469)', 'n (소득2-4)', '소득 2~4분위 (n=3,572)']
display(final_table)

,변수,n (전체),"전체 (n=5,041)",n (소득1),"소득 1분위 (n=1,469)",n (소득2-4),"소득 2~4분위 (n=3,572)"
0,**연령**,,,,,,
1,연령 (세),5041,48.62 (0.17),1469,55.68 (0.36),3572,46.40 (0.18)
2,65세 이상,1434,15.78,685,30.19,749,11.25
3,**지역**,,,,,,
4,읍면 거주,1995,22.50,784,34.22,1211,18.81
5,**교육 수준 및 신체계측**,,,,,,
6,초졸 이하,413,5.27,251,12.37,162,3.04
7,중졸,1155,19.52,534,33.97,621,14.99
8,고졸,1761,35.82,436,33.67,1325,36.49
9,대졸 이상,1633,39.39,223,19.99,1410,45.48
